In [1]:
%pip install openpyxl


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ==============================================================================
# NOTEBOOK 1: NETTOYAGE ET STRUCTURATION DES MÉTADONNÉES EXCEL
# PROJET: Text Mining - L'Enquêteur du Niger
# ==============================================================================
import pandas as pd
import os

# 1. Configuration des chemins
EXCEL_INPUT_PATH = "../data/raw/enqueteur.xlsx"  # Vérifie bien l'extension (.xlsx ou .xls)
CLEANED_OUTPUT_PATH = "../data/processed/enqueteur_cleaned.csv"

print("--- Chargement du fichier Excel ---")
# Lecture du fichier Excel
df_raw = pd.read_excel(EXCEL_INPUT_PATH)
print(f"Dimensions initiales : {df_raw.shape[0]} lignes et {df_raw.shape[1]} colonnes.")



--- Chargement du fichier Excel ---
Dimensions initiales : 555 lignes et 300 colonnes.


In [3]:
# ==============================================================================
# ÉTAPE 1 : DICTIONNAIRE DE SÉLECTION ET DE RENOMMAGE
# ==============================================================================
# Ce dictionnaire isole uniquement les métadonnées stratégiques que nous avons validées
mapping_colonnes = {
    'postId': 'post_id',
    'time': 'date_utc',
    'timestamp': 'timestamp',
    'text': 'texte',
    'url': 'url_post',
    'comments': 'nb_commentaires',
    'shares': 'nb_partages',
    'reactionLikeCount': 'like',
    'reactionLoveCount': 'love',
    'reactionCareCount': 'solidaire',
    'reactionHahaCount': 'haha',
    'reactionWowCount': 'wow',
    'reactionSadCount': 'triste',
    'reactionAngryCount': 'colere',
    'sharedPost/text': 'texte_partage',
    'sharedPost/pageName/name': 'nom_page_partage'
}

# Filtrage : On ne prend que les colonnes qui existent réellement dans ton fichier
colonnes_cibles = [col for col in mapping_colonnes.keys() if col in df_raw.columns]
df = df_raw[colonnes_cibles].rename(columns=mapping_colonnes).copy()

print(f"Dimensions après conservation des métadonnées clés : {df.shape}")

Dimensions après conservation des métadonnées clés : (555, 16)


In [4]:

# ==============================================================================
# ÉTAPE 2 : NETTOYAGE DES LIGNES VIDES ET DOUBLONS
# ==============================================================================

# 1. Gestion des textes manquants
# On s'assure qu'au moins 'texte' ou 'texte_partage' contient du texte pour le text mining
initial_len = len(df)
df = df.dropna(subset=['texte', 'texte_partage'], how='all')
print(f"Lignes supprimées (aucun texte exploitable) : {initial_len - len(df)}")

# En cas de valeur manquante sur l'un des deux, on remplace par une chaîne vide pour éviter les bugs futurs
df['texte'] = df['texte'].fillna('').astype(str).str.strip()
df['texte_partage'] = df['texte_partage'].fillna('').astype(str).str.strip()

# Supprimer les lignes qui ne contiennent que des espaces ou chaînes vides après strip
df = df[(df['texte'] != '') | (df['texte_partage'] != '')]

# 2. Suppression des doublons basés sur l'ID du post
if 'post_id' in df.columns:
    len_avant_doublons = len(df)
    df = df.drop_duplicates(subset=['post_id'])
    print(f"Doublons supprimés (basé sur post_id) : {len_avant_doublons - len(df)}")


Lignes supprimées (aucun texte exploitable) : 216
Doublons supprimés (basé sur post_id) : 0


In [5]:
# ==============================================================================
# ÉTAPE 3 : HARMONISATION DES METRICS ET DES DATES
# ==============================================================================

# Remplissage des valeurs manquantes pour les compteurs d'engagements par 0
colonnes_metriques = ['nb_commentaires', 'nb_partages', 'like', 'love', 'solidaire', 'haha', 'wow', 'triste', 'colere']
for col in colonnes_metriques:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

# Parsing de la date
if 'date_utc' in df.columns:
    df['date_utc'] = pd.to_datetime(df['date_utc'], errors='coerce')
    df = df.dropna(subset=['date_utc'])
    print(f"Plage temporelle de l'étude : du {df['date_utc'].min()} au {df['date_utc'].max()}")

print(f"\nDimensions finales du dataset propre : {df.shape}")

Plage temporelle de l'étude : du 2025-05-12 17:26:14+00:00 au 2026-07-14 22:33:26+00:00

Dimensions finales du dataset propre : (339, 16)


In [6]:

# ==============================================================================
# ÉTAPE 4 : SAUVEGARDE
# ==============================================================================
os.makedirs(os.path.dirname(CLEANED_OUTPUT_PATH), exist_ok=True)
df.to_csv(CLEANED_OUTPUT_PATH, index=False, encoding='utf-8')
print(f"Fichier nettoyé sauvegardé avec succès dans : {CLEANED_OUTPUT_PATH}")

Fichier nettoyé sauvegardé avec succès dans : ../data/processed/enqueteur_cleaned.csv
